# Verify the visualization pyramid

Acceptance gates for a pyramid built by [`build_pyramid.py`](build_pyramid.py), plus the
final `Cache-Control` pass. Run against the scratch prefix after a shakedown or the
production prefix after a `build_pyramid.yml` run — set `PYRAMID_PREFIX` below. Everything
up to the last section is read-only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import zarr

from build_pyramid import JOBS, dest_store, open_source
from global_snowmelt_runoff_onset.config import Config
from global_snowmelt_runoff_onset import store as gsro_store

config = Config('config/global_config_v10.txt')

# prefix + source tag come from the config (the single knob for version,
# generation, and tag); override PYRAMID_PREFIX for scratch shakedowns.
# To verify a legacy pyramid (e.g. v9, issue #13) just flip the config
# filename: release_tag is None there, and open_source falls back to the
# frozen Zarr v2 store with its .zmetadata ETag as the snapshot id.
PYRAMID_PREFIX = config.global_runoff_multiscale_azure_prefix
# PYRAMID_PREFIX = 'snowmelt/snowmelt_runoff_onset/scratch_pyramid_shakedown'
SOURCE_TAG = config.release_tag
N_LEVELS = 10

root = zarr.open_group(dest_store(config, PYRAMID_PREFIX, read_only=True), mode='r')
expected_vars = sorted(v for job in JOBS.values() for v in job
                       if v in root['0'].array_keys())
print(f'variables present: {expected_vars}')
missing_vars = sorted(set(v for job in JOBS.values() for v in job) - set(expected_vars))
if missing_vars:
    print(f'WARNING: job variables not in the store (partial build?): {missing_vars}')

## 1. Structure and attrs lint

Levels exist with iterated floor-halved shapes, the convention attrs are present
(`zarr_conventions`, `multiscales` layout, `proj:`, `spatial:`, `layer_hints`, `provenance`),
and every array carries the right `_FillValue` at both the attr and zarr `fill_value` level —
the v10 store-init bug class, asserted forever.

In [ ]:
attrs = dict(root.attrs)
for key in ['zarr_conventions', 'multiscales', 'proj:code', 'spatial:transform',
            'spatial:shape', 'provenance']:
    assert key in attrs, f'missing root attr: {key}'
assert attrs['proj:code'] == 'EPSG:4326'
prov = attrs['provenance']
print(f"built from {prov['source_store']} @ {prov['source_tag'] or 'no tag (legacy store)'} "
      f"(snapshot {prov['source_snapshot_id']}), topozarr {prov['topozarr_version']}")

layout = attrs['multiscales']['layout']
assert [entry['asset'] for entry in layout] == [str(i) for i in range(N_LEVELS)]

# level-0 shape from the config's realized geobox (v10: 204800 x 499998,
# v9: 195970 x 499998) -- the same gate verifies either era's pyramid
ny, nx = (int(v) for v in config.global_geobox.shape.yx)
for i in range(N_LEVELS):
    level = root[str(i)]
    for var in expected_vars:
        arr = level[var]
        assert arr.shape[-2:] == (ny, nx), (i, var, arr.shape)
        assert arr.dtype == np.int16
        assert arr.fill_value == -9999, f'{i}/{var} zarr fill_value {arr.fill_value}'
        assert arr.attrs['_FillValue'] == -9999
    ny, nx = ny // 2, nx // 2
print(f'{N_LEVELS} levels x {len(expected_vars)} vars: shapes, dtypes, fills all OK')

## 2. Level 0 vs source — exact

Raw (encoded) equality between pyramid level 0 and the tagged icechunk store on the QC tiles
from `3_quality_check_tiles.ipynb` — level 0 is a copy, so any mismatch is a bug, not a
tolerance question. (Uses one dense water year for the yearly variables.
`seasonal_snow_pct` is skipped here — it's reclassified from the Sturm & Liston GeoTIFF,
not copied from the icechunk store, and gets its own value-domain gate in section 2b.)

In [ ]:
# Sample windows for the byte-identity check, on the CONFIG's own tile
# grid. The slices hit source and level 0 identically, so the check is
# valid on any era's grid (on v9 these land 2 tile rows south of their
# v10 namesakes -- see store.grid_pixel_offset).
QC_TILES = [(25, 39), (9, 138), (10, 0), (28, 65), (80, 240)]
CHECK_WY = 2020

source_ds, snapshot_id = open_source(config, SOURCE_TAG)
assert snapshot_id == prov['source_snapshot_id'], 'pyramid was built from a different snapshot'
level0 = xr.open_zarr(dest_store(config, PYRAMID_PREFIX, read_only=True),
                      group='0', zarr_format=3, consolidated=False,
                      mask_and_scale=False, chunks=None)

# seasonal_snow_pct comes from the snow-class GeoTIFF, not the icechunk store
icechunk_vars = [v for v in expected_vars if v != 'seasonal_snow_pct']
for row, col in QC_TILES:
    region = gsro_store.tile_region_slices(config, row, col)
    for var in icechunk_vars:
        src = source_ds[var]
        pyr = level0[var]
        if 'water_year' in src.dims:
            src, pyr = src.sel(water_year=CHECK_WY), pyr.sel(water_year=CHECK_WY)
        src_vals = src.isel(**region).values
        pyr_vals = pyr.isel(**region).values
        assert (src_vals == pyr_vals).all(), f'level-0 mismatch: tile ({row},{col}) {var}'
    n_valid = int((src_vals != -9999).sum())
    print(f'tile ({row},{col}): all vars byte-identical ({n_valid:,} valid px in {var})')

## 2b. `seasonal_snow_pct` value domain

The mask is reclassified from the Sturm & Liston (2021) snow classification (NSIDC-0768)
rather than copied from the icechunk store, so instead of source equality it gets a domain
gate: level 0 must be **exactly** {0, 100, -9999} (accepted classes → 100, Ephemeral → 0,
ocean/fill → -9999); coarser levels are fill-aware integer means of those, so any value in
[0, 100] ∪ {-9999} — the percent of seasonal-snow area among classified land.

In [ ]:
MASK_VAR = 'seasonal_snow_pct'
if MASK_VAR not in expected_vars:
    print(f'{MASK_VAR} not in the store -- skipping (scratch shakedown?)')
else:
    # level 0: exact {0, 100, -9999} on the QC tiles
    for row, col in QC_TILES:
        region = gsro_store.tile_region_slices(config, row, col)
        vals = level0[MASK_VAR].isel(**region).values
        assert np.isin(vals, [0, 100, -9999]).all(), \
            f'level-0 domain violation, tile ({row},{col}): {np.unique(vals)[:12]}'
        print(f'tile ({row},{col}): level-0 values exactly {{0, 100, -9999}} '
              f'({100 * (vals == 100).mean():.1f}% seasonal snow)')

    # levels > 0: [0, 100] plus fill -- QC-tile windows while the arrays are
    # big, whole arrays once they're small
    for lvl in range(1, N_LEVELS):
        ds_l = xr.open_zarr(dest_store(config, PYRAMID_PREFIX, read_only=True),
                            group=str(lvl), zarr_format=3, consolidated=False,
                            mask_and_scale=False, chunks=None)
        if lvl <= 6:
            factor = 2 ** lvl
            windows = [{k: slice(s.start // factor, s.stop // factor)
                        for k, s in gsro_store.tile_region_slices(config, row, col).items()}
                       for row, col in QC_TILES]
            vals = np.concatenate([ds_l[MASK_VAR].isel(**w).values.ravel()
                                   for w in windows])
        else:
            vals = ds_l[MASK_VAR].values.ravel()
        assert ((vals == -9999) | ((vals >= 0) & (vals <= 100))).all(), \
            f'level-{lvl} domain violation: {np.unique(vals)[:12]}'
        print(f'level {lvl}: values in [0, 100] + fill OK '
              f'({int((vals != -9999).sum()):,} valid px checked)')

## 3. Cross-level visuals

Decoded views of one region at native / mid / coarse levels — the coarsening should look like
a blur of the same field, never a shift, a striping pattern, or values outside the children's
range. Rainier tile (25,39); swap in any region.

In [ ]:
row, col = 25, 39
var = 'runoff_onset_median' if 'runoff_onset_median' in expected_vars else expected_vars[0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, lvl in zip(axes, [0, 2, 4]):
    factor = 2 ** lvl
    ds_l = xr.open_zarr(dest_store(config, PYRAMID_PREFIX, read_only=True),
                        group=str(lvl), zarr_format=3, consolidated=False, chunks=None)
    da = ds_l[var]
    if 'water_year' in da.dims:
        da = da.sel(water_year=CHECK_WY)
    region = gsro_store.tile_region_slices(config, row, col)
    sub = da.isel(latitude=slice(region['latitude'].start // factor,
                                 region['latitude'].stop // factor),
                  longitude=slice(region['longitude'].start // factor,
                                  region['longitude'].stop // factor))
    sub.plot.imshow(ax=ax, vmin=110, vmax=270, cmap='viridis', add_colorbar=(lvl == 4))
    ax.set_title(f'level {lvl} ({80 * factor:.0f} m)')
    ax.set_aspect('equal')
fig.suptitle(f'{var}, tile ({row},{col})')
fig.tight_layout()

## 4. Global sanity render

The static-map preview: one coarse level, whole world, decoded. This is essentially what the
`../global/` figure notebooks will consume instead of the old coarsened store.

In [ ]:
ds_l6 = xr.open_zarr(dest_store(config, PYRAMID_PREFIX, read_only=True),
                     group='6', zarr_format=3, consolidated=False, chunks="auto")
da = ds_l6[var]
if 'water_year' in da.dims:
    da = da.sel(water_year=CHECK_WY)
da = da.compute()

fig, ax = plt.subplots(figsize=(16, 6))
da.plot.imshow(ax=ax, vmin=110, vmax=270, cmap='viridis')
ax.set_title(f'{var} @ level 6')
print(f'valid px at level 6: {int(da.notnull().sum()):,}')

## 5. Cache headers

`Cache-Control: public, max-age=31536000, immutable` on every blob under the prefix — safe
because cache-busting is by prefix version, not by mutation. Re-run after any job rerun.
**Writes blob properties**; everything above this cell is read-only.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from azure.storage.blob import ContainerClient, ContentSettings

import logging
logging.getLogger('azure').setLevel(logging.WARNING)

container, prefix = PYRAMID_PREFIX.split('/', 1)
account_url = f'https://{config.azure_storage_account}.blob.core.windows.net'
client = ContainerClient(account_url, container, credential=config.sas_token)

CACHE = 'public, max-age=31536000, immutable'
blobs = [b for b in client.list_blobs(name_starts_with=prefix + '/')]
todo = [b for b in blobs if (b.content_settings.cache_control or '') != CACHE]
print(f'{len(blobs):,} blobs, {len(todo):,} need the header')


def _set_header(blob):
    settings = ContentSettings(cache_control=CACHE,
                               content_type=blob.content_settings.content_type)
    client.get_blob_client(blob.name).set_http_headers(content_settings=settings)


with ThreadPoolExecutor(max_workers=16) as pool:
    list(pool.map(_set_header, todo))
print('done')